# DreamNest RAG Evaluation

I. Evaluate naive (semantic-only) retriever

In [ ]:
import sys
import os
from pathlib import Path
# In notebook, cwd is evals/ so project root is its parent
project_root = Path.cwd().parent
os.chdir(project_root)          # change working dir to project root
sys.path.append(str(project_root))

# import funcitons etc from agent.py
import agent

from typing import Dict, Any
from datasets import Dataset

# import ragas functions and metrics
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

In [ ]:
# Initialise agent pipeline with vector store and llm
vector_store = agent.init_pipeline()
llm = agent.get_llm()

In [ ]:
# Manually defined small golden test set (dream queries and reference answers)
# explicitly test key behsviours 
golden_cases = [
    {
        "question": "Have I dreamt about a house before?",
        "reference": "Yes, you have dreamt about houses before, including a childhood house and a small wooden house.",
    },
    {
        "question": "Have I dreamt about whales before?",
        "reference": "No, there are no dreams featuring whales in your archive.",
    },
    {
        "question": "Have I dreamt about fire before?",
        "reference": "Yes, you have dreamt about fire, for example the forest clearing with a fire (Dream 6).",
    },
    {
        "question": "Have I dreamt about penguins before?",
        "reference": "No, there are no dreams featuring penguins in your archive.",
    },
    {
        "question": "What recurring locations appear in my dreams?",
        "reference": "You often dream about houses, bodies of water (lakes, rivers, sea, pools), and bridges.",
    },
]

In [ ]:
def run_pipeline(question: str) -> Dict[str, Any]:
    """ Uses same logic as the backend Returns both the final answer and the raw retrieved contexts (top-k semantic).
    """
    answer = agent.answer_dream_query(question, vector_store, llm)

    # For evaluation, also capture the raw retrieved chunks (before LLM)
    docs_with_scores = vector_store.similarity_search_with_score(question, k=3)
    contexts = [doc.page_content for doc, score in docs_with_scores]

    return {"answer": answer, "contexts": contexts}


records = []
for case in golden_cases:
    q = case["question"]
    gt = case["reference"]

    result = run_pipeline(q)
    records.append(
        {
            "question": q,
            "answer": result["answer"],
            "contexts": result["contexts"],
            "ground_truth": gt,
        }
    )

baseline_dataset = Dataset.from_list(records)
baseline_dataset

In [ ]:
# Evaluate baseline retriever with RAGAS using OpenAI evaluator
# Set OPENAI_API_KEY in your environment before running this cell.

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

baseline_results = evaluate(baseline_dataset, metrics)
baseline_results

older code

Create synthetic testset with Ragas using abstracted approach for retriever evaluation
load data, chunk,  rebuilt the vector database 
  define retriever 

In [ ]:
# load data
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/")
raw_docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=0)
dreams_docs = text_splitter.split_documents(raw_docs)

print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(dreams_docs)}")
print(f"\nExample chunk:\n{dreams_docs[0]}")

In [ ]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from ragas.testset import TestsetGenerator


generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# generate testset
testset = generator.generate_with_langchain_docs(dreams_docs, testset_size=15)

testset.to_pandas()

In [ ]:
# define evalution metrics

from ragas import EvaluationDataset
from ragas.metrics import LLMContextRecall, ContextEntityRecall, NoiseSensitivity, ContextPrecision 
from ragas import evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper

# define metrics
retriever_metrics = [
    LLMContextRecall(),
    ContextPrecision(),
    ContextEntityRecall(),
    NoiseSensitivity(),
]

# define run config
run_config = RunConfig(timeout=360)
# define evaluator
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

In [ ]:
# get context and response from naive retriever
for test_row in tesset:
  response = naive_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"].content
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]